Ingestation


Installing the libraries

In [0]:
%pip install -q pypdf pandas databricks-vectorsearch

In [0]:
from pathlib import Path
from typing import List, Dict
import re
import pandas as pd
from pypdf import PdfReader

def clean_text(text: str) -> str:
    text = text.replace('\x00', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def normalize_text(text: str) -> str:
    text = clean_text(text)
    text = text.replace('"', '\"').replace('"', '\"').replace(''', "'").replace(''', "'")
    return text

In [0]:
def extract_pdf_pages(pdf_path: Path) -> List[Dict]:
    reader = PdfReader(str(pdf_path))
    rows = []
    for page_num, page in enumerate(reader.pages, start=1):
        raw = page.extract_text() or ""
        text = normalize_text(raw)  # Fixed: was 'text', should be 'raw'
        if not text:
            continue
        rows.append({
            "source_file": pdf_path.name,
            "title": pdf_path.stem,
            "page": page_num,
            "doc_type": "pdf",
            "text": text,
        })
    return rows

In [0]:
def extract_csv_rows(csv_path: Path, text_columns: List[str] = None) -> List[Dict]:
    """
    Extract rows from CSV file. Each row becomes a document.
    
    Args:
        csv_path: Path to CSV file
        text_columns: List of column names to combine as text. If None, uses all columns.
    """
    try:
        df = pd.read_csv(csv_path)
        rows = []
        
        if text_columns is None:
            text_columns = df.columns.tolist()
        
        for idx, row in df.iterrows():
            text_parts = []
            for col in text_columns:
                if col in df.columns and pd.notna(row[col]):
                    text_parts.append(f"{col}: {row[col]}")
            
            text = normalize_text(" | ".join(text_parts))
            
            if not text:
                continue
            
            rows.append({
                "source_file": csv_path.name,
                "title": csv_path.stem,
                "page": int(idx) + 1,
                "doc_type": "csv",
                "text": text
            })
        
        return rows
    except Exception as e:
        print(f"Error processing CSV {csv_path}: {e}")
        return []

In [0]:
def extract_parquet_rows(parquet_path: Path, text_columns: List[str] = None) -> List[Dict]:
    """
    Extract rows from Parquet file. Each row becomes a document.
    
    Args:
        parquet_path: Path to Parquet file
        text_columns: List of column names to combine as text. If None, uses all columns.
    """
    try:
        # Read parquet file using pandas
        df = pd.read_parquet(parquet_path)
        rows = []
        
        # If no text columns specified, use all columns
        if text_columns is None:
            text_columns = df.columns.tolist()
        
        for idx, row in df.iterrows():
            # Combine specified columns into text
            text_parts = []
            for col in text_columns:
                if col in df.columns and pd.notna(row[col]):
                    text_parts.append(f"{col}: {row[col]}")
            
            text = normalize_text(" | ".join(text_parts))
            
            if not text:
                continue
            
            rows.append({
                "source_file": parquet_path.name,
                "title": parquet_path.stem,
                "page": int(idx) + 1,  # Use row number as page
                "doc_type": "parquet",
                "text": text
            })
        
        return rows
    except Exception as e:
        print(f"Error processing Parquet {parquet_path}: {e}")
        return []

In [0]:
def load_corpus(raw_dir: str, csv_text_columns: List[str] = None, parquet_text_columns: List[str] = None) -> List[Dict]:
    """
    Load documents from multiple formats: PDF, CSV, Parquet, TXT.
    Each format is processed differently:
    - PDF: Extract text page by page
    - CSV: Each row becomes a document
    - Parquet: Each row becomes a document
    - TXT: Entire file as one document
    
    Args:
        raw_dir: Directory containing documents
        csv_text_columns: Columns to use from CSV files. If None, uses all columns.
        parquet_text_columns: Columns to use from Parquet files. If None, uses all columns.
    """
    raw_dir = Path(raw_dir)
    all_docs = []

    # Process PDF files
    pdf_files = list(raw_dir.rglob("*.pdf"))
    if pdf_files:
        print(f"Processing {len(pdf_files)} PDF files...")
        for pdf_path in pdf_files:
            docs = extract_pdf_pages(pdf_path)
            all_docs.extend(docs)
            print(f"  ✓ {pdf_path.name}: {len(docs)} pages")

    # Process Parquet files
    parquet_files = list(raw_dir.rglob("*.parquet"))
    if parquet_files:
        print(f"Processing {len(parquet_files)} Parquet files...")
        for parquet_path in parquet_files:
            docs = extract_parquet_rows(parquet_path, parquet_text_columns)
            all_docs.extend(docs)
            print(f"  ✓ {parquet_path.name}: {len(docs)} rows")

    # Process CSV files
    csv_files = list(raw_dir.rglob("*.csv"))
    if csv_files:
        print(f"Processing {len(csv_files)} CSV files...")
        for csv_path in csv_files:
            docs = extract_csv_rows(csv_path, csv_text_columns)
            all_docs.extend(docs)
            print(f"  ✓ {csv_path.name}: {len(docs)} rows")

    # Process TXT files
    txt_files = list(raw_dir.rglob("*.txt"))
    if txt_files:
        print(f"Processing {len(txt_files)} TXT files...")
        for txt_path in txt_files:
            text = clean_text(txt_path.read_text(encoding="utf-8", errors="ignore"))
            if text:
                all_docs.append({
                    "source_file": txt_path.name,
                    "title": txt_path.stem,
                    "page": 1,
                    "doc_type": "txt",
                    "text": text,
                })
                print(f"  ✓ {txt_path.name}")

    print(f"\nTotal documents loaded: {len(all_docs)}")
    return all_docs

In [0]:
def split_into_chunks(text: str, chunk_size: int = 2200, overlap: int = 300) -> List[str]:
    words = text.split()
    if not words:
        return []

    chunks = []
    start = 0
    n = len(words)

    while start < n:
        end = min(n, start + chunk_size)
        chunk = ' '.join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)

    return chunks

def build_chunks(docs: List[Dict]) -> List[Dict]:
    """
    Build chunks from documents. Handles all document types.
    Note: row_data is NOT stored to avoid Spark serialization issues.
    """
    chunk_rows = []
    for doc in docs:
        chunks = split_into_chunks(doc['text'])
        for i, chunk in enumerate(chunks):
            chunk_data = {
                'chunk_id': f"{doc['source_file']}-{doc['page']}-{i}",
                'source_file': doc['source_file'],
                'title': doc['title'],
                'page': doc['page'],
                'doc_type': doc['doc_type'],
                'text': chunk
            }
            
            chunk_rows.append(chunk_data)
    
    return chunk_rows

## Store Each File in Separate Delta Tables

This pipeline now:
* **Separate Storage**: Each file gets its own Delta table (e.g., `ipc_chunks`, `bns_chunks`, `constitution_chunks`, `train_00001_chunks`)
* **Unified Vector Search**: Creates a view that unions all tables, materializes it, then builds one vector index for searching across all documents
* **Multi-Format Support**: PDF, CSV, Parquet, TXT files all processed with their specific pipelines

**Supported Formats:**
* **PDF** → Page-by-page extraction (legal documents, regulations)
* **Parquet** → Row-by-row processing (structured datasets)
* **CSV** → Row-by-row processing (case records, legal data)
* **TXT** → Full file extraction (plain text documents)

**Benefits:**
* Query individual tables when you need file-specific data
* Search across ALL documents using a single vector index
* Update files independently without affecting others

In [0]:
import os
from pyspark.sql.functions import col, concat_ws, lit, monotonically_increasing_id

# Configuration
catalog = "workspace"
schema = "default"

# List of file paths to process
raw_documents_paths = [
    "/Volumes/workspace/default/legal_docs/bns_sections.csv",
    "/Volumes/workspace/default/legal_docs/20240716890312078.pdf",
    "/Volumes/workspace/default/legal_docs/train-00000-of-00002-add4caaf8fbc6a8c.parquet",
    "/Volumes/workspace/default/legal_docs/train-00001-of-00002-09ac6bd45d6b3658.parquet",
]

# Optional: Specify which columns to use for text
csv_columns = None
parquet_columns = None

# Track all created tables
created_tables = []

print("Processing files...\n")

for file_path in raw_documents_paths:
    file_path_obj = Path(file_path)
    extension = file_path_obj.suffix.lower()
    table_name = re.sub(r'[^a-zA-Z0-9_]', '_', file_path_obj.stem.lower()) + "_chunks"
    full_table_name = f"{catalog}.{schema}.{table_name}"
    
    print(f"• {file_path_obj.name}", end=" ")
    
    if extension == '.pdf':
        docs = extract_pdf_pages(file_path_obj)
        if not docs:
            print("⚠ No data")
            continue
        chunks = build_chunks(docs)
        df = spark.createDataFrame(chunks)
        print(f"→ {len(chunks)} chunks")
    
    elif extension in ['.parquet', '.csv']:
        # Use Spark directly for structured files
        df_raw = spark.read.parquet(file_path) if extension == '.parquet' else \
                 spark.read.option("header", "true").option("inferSchema", "true").csv(file_path)
        
        cols_to_use = parquet_columns if extension == '.parquet' else csv_columns
        if cols_to_use is None:
            cols_to_use = df_raw.columns
        
        # Combine columns into text and add metadata
        text_expr = concat_ws(" | ", *[concat_ws(": ", lit(c), col(c).cast("string")) for c in cols_to_use])
        
        df = df_raw.select(
            text_expr.alias("text"),
            lit(file_path_obj.name).alias("source_file"),
            lit(file_path_obj.stem).alias("title"),
            monotonically_increasing_id().alias("page"),
            lit(extension[1:]).alias("doc_type")
        ).withColumn(
            "chunk_id",
            concat_ws("-", col("source_file"), col("page"), lit("0"))
        )
        
        row_count = df.count()
        print(f"→ {row_count} rows")
    
    elif extension == '.txt':
        text = clean_text(file_path_obj.read_text(encoding="utf-8", errors="ignore"))
        if not text:
            print("⚠ No data")
            continue
        docs = [{"source_file": file_path_obj.name, "title": file_path_obj.stem, "page": 1, "doc_type": "txt", "text": text}]
        chunks = build_chunks(docs)
        df = spark.createDataFrame(chunks)
        print(f"→ {len(chunks)} chunks")
    
    else:
        print(f"⚠ Unsupported")
        continue
    
    df.write.mode("overwrite").saveAsTable(full_table_name)
    created_tables.append(full_table_name)

print(f"\n✓ Created {len(created_tables)} tables")
for table in created_tables:
    print(f"  {table}")

In [0]:
# Create a unified view that unions all the separate tables
view_name = f"{catalog}.{schema}.legal_docs_unified_view"

if created_tables:
    # Build UNION ALL query with standardized schema
    union_parts = []
    for table in created_tables:
        union_parts.append(f"""
            SELECT text, source_file, title, CAST(page AS BIGINT) AS page, doc_type, chunk_id
            FROM {table}
        """)
    
    union_query = " UNION ALL ".join(union_parts)
    
    spark.sql(f"CREATE OR REPLACE VIEW {view_name} AS {union_query}")
    
    print(f"✓ Created unified view: {view_name}")
    print(f"  Combines {len(created_tables)} tables\n")
    
    print("Sample:")
    display(spark.table(view_name).limit(3))
else:
    print("⚠ No tables to union")

In [0]:
# Example: If you have different types of CSV files and want to specify columns for each
# Uncomment and modify as needed:

"""
# Option 1: For legal case records CSV with specific columns
csv_columns_case_records = ['case_number', 'case_title', 'judgment_text', 'court', 'date']
docs_cases = load_corpus('/path/to/case_records/', csv_text_columns=csv_columns_case_records)

# Option 2: For legal sections/articles CSV
csv_columns_sections = ['section_number', 'title', 'description', 'penalties']
docs_sections = load_corpus('/path/to/sections/', csv_text_columns=csv_columns_sections)

# Option 3: For precedent/citations CSV
csv_columns_citations = ['citation', 'summary', 'court', 'year']
docs_citations = load_corpus('/path/to/citations/', csv_text_columns=csv_columns_citations)

# Combine all documents
all_docs = docs_cases + docs_sections + docs_citations

# Build chunks and process
all_chunks = build_chunks(all_docs)
df_combined = spark.createDataFrame(all_chunks)
df_combined.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.{table_name}")
"""

## Generate Embeddings and Create Vector Search Index

For RAG, you need to:
1. Generate embeddings for each text chunk
2. Store embeddings in a Delta table
3. Create a Vector Search index

In [0]:
from databricks.vector_search.client import VectorSearchClient

catalog = "workspace"
schema = "default"
materialized_table = f"{catalog}.{schema}.legal_docs_all_chunks"

print(f"Materializing table: {materialized_table}\n")

# Get tables to union
try:
    tables_to_union = created_tables
except NameError:
    tables_df = spark.sql(f"SHOW TABLES IN {catalog}.{schema}")
    tables_to_union = [
        f"{catalog}.{schema}.{row.tableName}" 
        for row in tables_df.collect() 
        if row.tableName.endswith('_chunks') and row.tableName != 'legal_docs_all_chunks'
    ]

if not tables_to_union:
    print("⚠ No chunk tables found. Run Cell 11 first.")
else:
    # Build UNION ALL and create table directly
    union_parts = []
    for table in tables_to_union:
        union_parts.append(f"SELECT text, source_file, title, CAST(page AS BIGINT) AS page, doc_type, chunk_id FROM {table}")
    
    union_query = " UNION ALL ".join(union_parts)
    
    spark.sql(f"CREATE OR REPLACE TABLE {materialized_table} AS {union_query}")
    spark.sql(f"ALTER TABLE {materialized_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    
    print(f"✓ Table created with CDC enabled")
    print(f"\nSample:")
    display(spark.table(materialized_table).limit(3))

In [0]:
# Initialize Vector Search client
vsc = VectorSearchClient()

endpoint_name = "legal_rag_endpoint"
index_name = f"{catalog}.{schema}.legal_docs_vector_index"

print(f"Creating vector index: {index_name}\n")

# Create endpoint if needed
try:
    vsc.create_endpoint(name=endpoint_name)
    print(f"✓ Created endpoint: {endpoint_name}")
except Exception as e:
    print(f"Using existing endpoint: {endpoint_name}")

# Create vector index
try:
    index = vsc.create_delta_sync_index(
        endpoint_name=endpoint_name,
        index_name=index_name,
        source_table_name=materialized_table,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="text",
        embedding_model_endpoint_name="databricks-bge-large-en"
    )
    print(f"✓ Vector index created: {index_name}")
    print(f"\nIndex ready for similarity search across all documents")
except Exception as e:
    print(f"⚠ Error: {e}")

## Query the Vector Database for RAG

Now you can search for similar documents using natural language queries

## Complete Multi-Format RAG Pipeline Summary

### Storage Architecture:
* **Each file → Separate Delta table** (PDF, Parquet, CSV, TXT files)
* **Unified vector index** searches across ALL documents

### Supported Formats:
* **PDF** → Page-by-page text extraction (legal documents, regulations)
* **Parquet** → Row-by-row processing (structured datasets)
* **CSV** → Row-by-row processing (case records, legal data)
* **TXT** → Full file extraction (plain text documents)

### Benefits:
* Query individual tables for file-specific data
* Single vector search across all legal documents
* Update files independently
* Handle both unstructured (PDF) and structured (Parquet/CSV) data

### Next Steps:
1. File paths are already configured in [Cell 11](#cell-6630bf71-80f6-4cc5-9103-99b391ee458a)
2. Run [Cell 11](#cell-6630bf71-80f6-4cc5-9103-99b391ee458a) to process all files and create tables
3. Run [Cell 12](#cell-125c0a59-8ff8-49e1-bae7-045383aafe13) to create unified view
4. Run [Cell 16](#cell-04c99e93-0efd-405e-88e2-66739c01bba1) to materialize for vector search
5. Run [Cell 17](#cell-3b995397-531a-4fb9-9310-218243932703) to create vector index
6. Test queries using [Cell 20](#cell-65ffbd65-df63-470b-8f09-352d7d716dd7)

In [0]:
# Example query for RAG
query = "What are the provisions related to theft in IPC?"

# Check if index is ready before searching
try:
    index = vsc.get_index(endpoint_name, index_name)
    status_info = index.describe()
    
    if not status_info.get('status', {}).get('ready', False):
        print("⚠ Vector index is still provisioning. This can take 5-10 minutes after creation.")
        print(f"   Current status: {status_info.get('status', {}).get('state', 'Unknown')}")
        print("\nPlease wait a few minutes and re-run this cell.")
    else:
        # Index is ready - perform search
        results = index.similarity_search(
            query_text=query,
            columns=["chunk_id", "source_file", "title", "page", "text"],
            num_results=5
        )
        
        # Display results
        results_df = pd.DataFrame(results.get('result', {}).get('data_array', []))
        print(f"Top {len(results_df)} results for query: '{query}'")
        display(results_df)
        
except Exception as e:
    print(f"Error: {e}")
    print("\nIf you see 'Something went wrong unexpectedly', the index may still be syncing.")
    print("Wait 5-10 minutes after creating the index, then try again.")

In [0]:
from databricks.vector_search.client import VectorSearchClient
import time

vsc = VectorSearchClient(disable_notice=True)
endpoint_name = "legal_rag_endpoint"
index_name = "workspace.default.legal_docs_vector_index"

print("🔍 Monitoring vector index status...\n")
max_checks = 1  # Check for up to 5 minutes (10 seconds * 30)

for i in range(max_checks):
    try:
        index = vsc.get_index(endpoint_name, index_name)
        status_info = index.describe()
        status = status_info.get('status', {})
        
        ready = status.get('ready', False)
        state = status.get('detailed_state', status.get('state', 'Unknown'))
        message = status.get('message', 'N/A')
        
        timestamp = time.strftime('%H:%M:%S')
        print(f"[{timestamp}] State: {state} | Ready: {ready}")
        
        if message != 'N/A':
            print(f"          Message: {message}")
        
        if ready:
            print("\n✅ Index is READY! You can now run queries.")
            print("\nRe-run Cell 19 to test similarity search.")
            break
        
        if i < max_checks - 1:
            time.sleep(10)  # Wait 10 seconds between checks
    
    except Exception as e:
        print(f"Error checking status: {e}")
        break
else:
    print("\n⏳ Index is still provisioning. Please wait a few more minutes and check again.")

In [0]:
# Example: Query a specific file's table directly
print("Example: Querying IPC table directly")
print("="*80)

# Query specific table (e.g., IPC only)
ipc_table = f"{catalog}.{schema}.ipc_chunks"
try:
    ipc_df = spark.table(ipc_table)
    print(f"\nIPC document chunks: {ipc_df.count()}")
    print("\nSample IPC chunks:")
    display(ipc_df.select('chunk_id', 'title', 'page', 'text').limit(3))
except:
    print(f"Table {ipc_table} not found. Update the table name to match your files.")

print("\n" + "="*80)
print("You can query any individual table:")
for table in created_tables:
    print(f"  - spark.table('{table}')")

## Architecture Summary

### Storage Layer:
```
workspace.default.train_00001_chunks          ← Parquet data
workspace.default.train_00000_chunks          ← Parquet data
workspace.default.bns_sections_chunks         ← CSV data
workspace.default.20240716890312078_chunks    ← PDF data
          ↓
workspace.default.legal_docs_unified_view  (SQL View - UNION ALL)
          ↓
workspace.default.legal_docs_all_chunks    (Materialized Delta Table)
```

### Vector Search Layer:
```
Vector Index: workspace.default.legal_docs_vector_index
  ↓
Searches across ALL files (PDF, Parquet, CSV, TXT) using semantic similarity
```

### Supported Formats:
* **PDF** → Page-by-page extraction
* **Parquet** → Row-by-row processing (structured datasets)
* **CSV** → Row-by-row processing  
* **TXT** → Full file as document

### Usage:
* **Individual file queries**: `spark.table('workspace.default.train_00001_chunks')`
* **Unified RAG search**: Use vector index to search across all documents
* **Updates**: Modify individual tables, then re-run Cell 12 & 16 to refresh unified table